# Semantic Memory in Agentic AI: The Agent's Knowledge Base

**Semantic memory = useful knowledge an agent can reuse beyond one conversation.**

| Kind | What it is | Shared? | How we retrieve it |
|---|---|---|---|
| **World knowledge** | Facts from documents, articles and manuals | Same for every user | Standard **RAG** |
| **User knowledge** | Facts about a particular user's background, skills and preferences | Personal | Fact extraction + **Mem0** |

This notebook demonstrates:

1. Setup
2. World knowledge: without RAG vs with RAG
3. A scripted conversation in which a user shares personal facts
4. What happens when the agent has no memory
5. Why plain similarity-based RAG can miss relevant personal facts
6. Mem0: Add, Search, Update and Delete
7. A final comparison

**Important implementation note for Groq's 8,000-token-per-minute limit**

Mem0's built-in extraction prompt can itself be larger than the free/on-demand Groq TPM allowance. To keep this notebook runnable on that tier, we use a short Groq prompt to extract concise facts, then store those facts in Mem0 using `infer=False`.

This means:
- Fact extraction is still performed by a real LLM call.
- Mem0 still stores, searches and deletes the facts using its vector store.
- The notebook does **not** demonstrate Mem0's built-in automatic extraction/update pipeline (`infer=True`). The Update step is performed explicitly by deleting the old fact and adding the new one.

LLM outputs can vary. The scripted user messages are test data; assistant replies and extracted facts are generated by real model calls.


# 1. Setup

In [1]:
%pip install -q langchain-groq sentence-transformers mem0ai

In [3]:
import os
import re
import json
import time
import shutil
import requests
import numpy as np
import pandas as pd

from google.colab import userdata
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer

# Add a Colab Secret named groq_api containing your Groq API key.
my_api_key = userdata.get("agentic_ai")
if not my_api_key:
    raise ValueError("Groq API key not found. Add a Colab Secret named 'groq_api'.")

os.environ["GROQ_API_KEY"] = my_api_key

# Main conversational model and compact fact-extraction model.
# Both use the same model to keep the setup simple.
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=my_api_key,
    max_retries=3
)

# Embeddings for world RAG and Mem0.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Setup complete.")


Setup complete.


## 1.1 Helpers

- `ask` sends one question to the LLM, with optional **memory text** placed in the system prompt, and returns the answer and the input tokens used.
- `rank` is plain RAG retrieval: it scores every text against the query by **cosine similarity** and returns them best first.

In [4]:
def ask(question, memory_text=None):
    """Ask one question. If memory_text is given, it is added to the system prompt."""
    system = "You are a helpful assistant."
    if memory_text:
        system += "\n\n" + memory_text

    reply = llm.invoke([
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ])
    return reply.content, reply.usage_metadata['input_tokens']

def rank(query, texts, vectors):
    """RAG retrieval: cosine similarity between the query and every text, best first."""
    q = embedder.encode(query)
    sims = vectors @ q / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
    order = np.argsort(-sims)
    return [(float(sims[i]), texts[i]) for i in order]

results = []   # one row per experiment, shown as a table at the end

# 2. World Knowledge: Standard RAG Works

World knowledge is the same for every user (articles, manuals, documentation). The LLM only knows what it was trained on, so for anything newer or private we **retrieve** the relevant text and put it in the prompt.

We use two real Wikipedia pages about a recent space mission, which is newer than the model's training data.

## 2.1 Load the real documents

In [5]:
def get_wikipedia(title):
    """Download the plain text of a Wikipedia page."""
    r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1,
                "titles": title, "format": "json", "redirects": 1},
        headers={"User-Agent": "colab-memory-demo/1.0"},
    )
    page = next(iter(r.json()["query"]["pages"].values()))
    return page["extract"]

documents = {title: get_wikipedia(title) for title in ["Axiom Mission 4", "Gaganyaan"]}

for title, text in documents.items():
    print(f"{title}: {len(text)} characters")

Axiom Mission 4: 13262 characters
Gaganyaan: 52798 characters


## 2.2 Chunk and embed (build the knowledge base)

We cut each page into chunks of roughly 800 characters and embed every chunk once. This index is shared by all users.

In [6]:
def chunk_text(text, size=800):
    """Group paragraphs into chunks of about `size` characters."""
    chunks, current = [], ""
    for paragraph in text.split("\n"):
        if not paragraph.strip():
            continue
        if len(current) + len(paragraph) > size and current:
            chunks.append(current.strip())
            current = ""
        current += paragraph + "\n"
    if current.strip():
        chunks.append(current.strip())
    return chunks

world_chunks = [c for text in documents.values() for c in chunk_text(text)]
world_vectors = embedder.encode(world_chunks)

print("Chunks in the knowledge base:", len(world_chunks))

Chunks in the knowledge base: 101


## 2.3 Without RAG: the LLM answers from training data only

In [7]:
world_question = "Which Indian astronaut flew on Axiom Mission 4, and which space station did the crew visit?"

answer, tokens = ask(world_question)
print("Tokens sent:", tokens)
print("Answer:", answer)

results.append({"experiment": "World knowledge | no RAG", "tokens sent": tokens,
                "check": "contains 'Shukla': " + str("shukla" in answer.lower())})

Tokens sent: 100
Answer: The Indian astronaut was **Rakesh Sharma**, and the crew flew to and worked aboard the **International Space Station (ISS)**.


## 2.4 With RAG: retrieve the top 3 chunks and add them to the prompt

In [8]:
ranked = rank(world_question, world_chunks, world_vectors)

print("Top 3 chunks by similarity:")
for score, chunk in ranked[:3]:
    print(f"  {score:.2f} | {chunk[:100]}...")

# Put the retrieved chunks into the prompt as memory
context = "Use this information to answer:\n\n" + "\n\n".join(chunk for _, chunk in ranked[:3])

answer, tokens = ask(world_question, context)
print("\nTokens sent:", tokens)
print("Answer:", answer)

results.append({"experiment": "World knowledge | RAG (top 3)", "tokens sent": tokens,
                "check": "contains 'Shukla': " + str("shukla" in answer.lower())})

Top 3 chunks by similarity:
  0.65 | Axiom Mission 4 (Ax‑4) was a private crewed spaceflight to the International Space Station (ISS) ope...
  0.64 | Ax‑4 represented a milestone for the Indian Human Spaceflight Programme, integrating with ISRO's Gag...
  0.63 | The four-person crew included commander Peggy Whitson, an Axiom Space employee and former NASA astro...

Tokens sent: 382
Answer: The Indian astronaut was **Shubhanshu Shukla**, and the crew flew to the **International Space Station (ISS)**.


**What to observe:**
- Without RAG, the model either admits it doesn't know or guesses from old training data.
- With RAG, the answer is grounded in the retrieved text.
- RAG works here because **the question and the chunk that answers it are semantically close**: the top chunks have high similarity scores and are about the same topic as the question.
- One index serves every user, so the cost of building it is shared.

# 3. User Knowledge: The User Tells the Agent About Themselves

User knowledge is different. It is **personal** and it **accumulates** across conversations. Here is Session 1 with a user called Riya. Her messages are scripted. The replies are real.

The agent is told to reply in one short sentence to keep the calls small.

In [9]:
# What the user says in Session 1 (scripted)
user_messages = [
    "Hi, I'm Riya. I work as an accountant at a mid-size firm in Pune.",
    "Python is a mystery to me. I've never written or run any Python and I don't want to install anything.",
    "I'm vegetarian, so no suggestions with meat please.",
    "I'm curious about how ChatGPT works. People keep saying it is built on something called transformers.",
    "I'm attending an AI conference in Bengaluru next month.",
    "I prefer short answers with simple analogies, no jargon.",
    "My team uses Excel for everything, pivot tables and VLOOKUP all day.",
    "I'm on a Windows laptop.",
]

# Session 1 conversation: this list IS the working memory of the session
session1 = [{"role": "system", "content": "You are a helpful assistant. Reply in one short sentence."}]

for text in user_messages:
    session1.append({"role": "user", "content": text})
    reply = llm.invoke(session1).content
    session1.append({"role": "assistant", "content": reply})
    print("User :", text)
    print("Agent:", reply, "\n")

User : Hi, I'm Riya. I work as an accountant at a mid-size firm in Pune.
Agent: Nice to meet you, Riya—how can I assist you today? 

User : Python is a mystery to me. I've never written or run any Python and I don't want to install anything.
Agent: Try an online Python sandbox like Replit or Google Colab—no installation needed. 

User : I'm vegetarian, so no suggestions with meat please.
Agent: Try an online Python sandbox like Replit or Google Colab—no installation needed. 

User : I'm curious about how ChatGPT works. People keep saying it is built on something called transformers.
Agent: ChatGPT is built on transformer neural networks that learn patterns from massive text data. 

User : I'm attending an AI conference in Bengaluru next month.
Agent: Enjoy the conference and make the most of the AI insights! 

User : I prefer short answers with simple analogies, no jargon.
Agent: Sounds exciting—think of it as a big tech party where ideas mingle like friends at a potluck. 

User : My t

**What to observe:** the user revealed a lot about herself, but only **one fact really matters** for a coding question: she doesn't know Python. Session 1 now ends and its working memory disappears.

# 4. User Knowledge Without Any Memory

A new session starts a few days later. The user asks the question from the notes.

In [10]:
user_question = "Give me code for Transformers"

answer, tokens = ask(user_question)
print("Tokens sent:", tokens)
print("Answer:", answer)

results.append({"experiment": "User knowledge | no memory", "tokens sent": tokens,
                "check": "gave a code block: " + str("```" in answer)})

Tokens sent: 85
Answer: Below is a **starter‑kit** that shows how to get up and running with the Hugging Face 🤗 Transformers library.  
It covers:

1. **Installation**  
2. **Inference** – text generation, summarization, and classification with a single line of code (`pipeline`).  
3. **Fine‑tuning** – a minimal training loop for a BERT‑based text‑classification task.  

Feel free to copy‑paste the snippets into a Jupyter notebook or a `.py` file and tweak them for your own data.

---

## 1. Install the library

```bash
# Create a fresh environment (optional but recommended)
python -m venv hf-env
source hf-env/bin/activate   # Windows: hf-env\Scripts\activate

# Install transformers + torch (CPU or GPU)
pip install --upgrade pip
pip install transformers torch  # add "torch==<desired_version>" if you need a specific CUDA version
```

> **Tip** – If you have an NVIDIA GPU, install the CUDA‑enabled PyTorch version from https://pytorch.org/ and then run `pip install transformers`.

---

##

**What to observe:** the agent has no idea who it is talking to, so it answers for a generic programmer. For this user, that answer is useless.

# 5. Why RAG Cannot Be Used for User Knowledge

Can we simply treat the user's messages like world knowledge? We embed them, retrieve the most similar ones to the new query, and add them to the prompt.

## 5.1 Build a RAG index over the user's messages

In [11]:
user_vectors = embedder.encode(user_messages)

ranked = rank(user_question, user_messages, user_vectors)

print(f"Query: {user_question}\n")
for position, (score, text) in enumerate(ranked, start=1):
    print(f"{position}. similarity {score:.2f} | {text}")

Query: Give me code for Transformers

1. similarity 0.44 | I'm curious about how ChatGPT works. People keep saying it is built on something called transformers.
2. similarity 0.13 | My team uses Excel for everything, pivot tables and VLOOKUP all day.
3. similarity 0.12 | I'm attending an AI conference in Bengaluru next month.
4. similarity 0.11 | I prefer short answers with simple analogies, no jargon.
5. similarity 0.11 | Python is a mystery to me. I've never written or run any Python and I don't want to install anything.
6. similarity 0.08 | I'm on a Windows laptop.
7. similarity 0.08 | Hi, I'm Riya. I work as an accountant at a mid-size firm in Pune.
8. similarity -0.04 | I'm vegetarian, so no suggestions with meat please.


## 5.2 Where does the important fact rank?

In [12]:
python_message = user_messages[1]   # the one fact that matters for this query

ranked_texts = [text for _, text in ranked]
print(f"The 'doesn't know Python' message ranks {ranked_texts.index(python_message) + 1} of {len(user_messages)}")

The 'doesn't know Python' message ranks 5 of 8


## 5.3 Answer using the top 2 retrieved messages

In [13]:
top_2 = ranked[:2]
print("Retrieved:")
for score, text in top_2:
    print(f"  {score:.2f} | {text}")

memory_text = "Facts the user told you (tailor your answer to them):\n" + "\n".join(f"- {t}" for _, t in top_2)

answer, tokens = ask(user_question, memory_text)
print("\nTokens sent:", tokens)
print("Answer:", answer)

results.append({"experiment": "User knowledge | RAG (top 2)", "tokens sent": tokens,
                "check": "gave a code block: " + str("```" in answer)})

Retrieved:
  0.44 | I'm curious about how ChatGPT works. People keep saying it is built on something called transformers.
  0.13 | My team uses Excel for everything, pivot tables and VLOOKUP all day.

Tokens sent: 135
Answer: Below is a **minimal, self‑contained implementation of a Transformer encoder block** written in PyTorch.  
It contains everything you need to understand the core mechanics (multi‑head self‑attention, layer‑norm, residuals, feed‑forward network) and can be dropped into a larger model (e.g., GPT‑style decoder, BERT‑style encoder, or a custom sequence‑to‑sequence network).

> **Why this code?**  
> * It is intentionally short (≈ 120 lines) so you can read it in a single pass.  
> * It uses only the standard PyTorch API – no external transformer libraries.  
> * It is fully differentiable and ready for training or inference.  
> * You can plug it into an Excel‑style pipeline by converting your tabular data into token indices (e.g., one‑hot or learned embeddings) and f

**What to observe:**
- RAG ranks by **similarity to the query**. The query is about "code" and "Transformers", so the messages that mention transformers and AI score highest.
- The fact that actually decides the answer ("doesn't know Python") has **no meaning-overlap with the query**, so it ranks low and is not retrieved.
- The user's facts are also scattered across raw messages, mixed with irrelevant ones (vegetarian, Windows).
- Retrieving more (a bigger `k`) only works by luck and grows the prompt with every conversation. For user knowledge we need to store **distilled facts** and decide relevance differently. That is what Mem0 does.

# 6. Mem0: A Memory Layer for User Knowledge

Mem0 sits between our conversations and the LLM. It has four operations:

| Operation | When | What it does |
|---|---|---|
| **Add** | End of a session | An LLM reads the conversation and stores the facts about the user |
| **Search** | Start of a session | Retrieves the user's memories for the new query |
| **Update** | When new info contradicts old | Rewrites the old memory instead of duplicating it |
| **Delete** | When a memory is expired or unwanted | Removes it |

## 6.1 Configure Mem0

Mem0 needs three things: an **LLM** (to extract facts), an **embedder** and a **vector store** (to save and search memories). We use Groq (a larger model than the one answering our questions, see the comment in the cell), the same embedding model as before, and a local Qdrant store, so no extra keys are needed.

> Run this cell **once**. The local store is locked while Mem0 uses it, so if you need to start over, restart the runtime.

In [14]:
from mem0 import Memory

# Start with an empty local store for a clean demonstration.
shutil.rmtree("./qdrant_db", ignore_errors=True)

config = {
    "llm": {
        "provider": "groq",
        "config": {
            # Mem0's internal LLM is not called by our infer=False writes.
            # The compact extraction call is made separately with `llm` above.
            "model": "openai/gpt-oss-20b",
            "temperature": 0,
            "api_key": my_api_key,
        },
    },
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": "all-MiniLM-L6-v2",
            "embedding_dims": 384,
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "collection_name": "user_memory",
            "embedding_model_dims": 384,
            "path": "./qdrant_db",
        },
    },
}

mem = Memory.from_config(config)
USER_ID = "riya"

print("Mem0 initialized for user:", USER_ID)


Mem0 initialized for user: riya


In [15]:
def memories(res):
    """Normalize Mem0 responses across SDK versions."""
    if isinstance(res, dict):
        return res.get("results", [])
    return res or []

def mem_get_all():
    """Get this user's memories; support current and older Mem0 signatures."""
    try:
        return mem.get_all(filters={"user_id": USER_ID})
    except TypeError:
        return mem.get_all(user_id=USER_ID)

def mem_search(query, top_k=10):
    """Search this user's memories; support current and older Mem0 signatures."""
    try:
        return mem.search(query, filters={"user_id": USER_ID}, top_k=top_k)
    except TypeError:
        return mem.search(query, user_id=USER_ID, limit=top_k)

def current_facts():
    """Return the text of all currently stored memories for this user."""
    return [item["memory"] for item in memories(mem_get_all()) if item.get("memory")]

def store_facts_without_inference(facts):
    """Store already-extracted facts in Mem0 without triggering its large LLM prompt."""
    fact_messages = [
        {"role": "user", "content": fact}
        for fact in facts
        if isinstance(fact, str) and fact.strip()
    ]
    if not fact_messages:
        print("No facts to store.")
        return None
    return mem.add(fact_messages, user_id=USER_ID, infer=False)

def extract_facts_compact(messages):
    """Use a short LLM prompt to turn a conversation into concise user facts."""
    conversation_text = "\n".join(
        f"- {m['content']}" for m in messages if m.get("role") == "user"
    )

    prompt = f"""Extract concise, useful facts about the user from the messages below.

Rules:
- Return only valid JSON in this format: {{"facts": ["fact 1", "fact 2"]}}
- Keep each fact short and self-contained.
- Preserve preferences, skills, location and relevant plans.
- Do not invent information.
- Do not include assistant statements.

User messages:
{conversation_text}
"""

    response = llm.invoke(prompt).content
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", response.strip(), flags=re.I)
    data = json.loads(cleaned)
    return [fact.strip() for fact in data.get("facts", []) if isinstance(fact, str) and fact.strip()]

print("Mem0 helper functions ready.")


Mem0 helper functions ready.


## 6.2 Add: extract compact facts, then store them in Mem0

The original Mem0 automatic extraction prompt can exceed the 8,000 TPM allowance. Here, a short Groq prompt extracts facts first. We then use `infer=False` so Mem0 stores those facts without making a second, oversized extraction call.

This keeps the distinction clear: **Groq extracts; Mem0 stores and retrieves.**


In [16]:
# Use only the user's messages from Session 1.
conversation = [m for m in session1 if m["role"] == "user"]

# Let the previous model calls age out of the one-minute TPM window.
time.sleep(60)

# Step 1: compact LLM-based fact extraction.
extracted_facts = extract_facts_compact(conversation)

print("Facts extracted by the LLM:")
for fact in extracted_facts:
    print(" -", fact)

# Step 2: store the extracted facts in Mem0 without its built-in inference prompt.
store_facts_without_inference(extracted_facts)

print("\\nMemories stored in Mem0:")
for fact in current_facts():
    print(" -", fact)


Facts extracted by the LLM:
 - Riya is an accountant at a mid-size firm in Pune.
 - She has never written or run Python and does not want to install anything.
 - She is vegetarian.
 - She wants a simple explanation of how ChatGPT works, especially transformers.
 - She will attend an AI conference in Bengaluru next month.
 - She prefers short answers with simple analogies and no jargon.
 - Her team relies heavily on Excel, using pivot tables and VLOOKUP.
 - She uses a Windows laptop.


\nMemories stored in Mem0:
 - Riya is an accountant at a mid-size firm in Pune.
 - Her team relies heavily on Excel, using pivot tables and VLOOKUP.
 - She wants a simple explanation of how ChatGPT works, especially transformers.
 - She is vegetarian.
 - She uses a Windows laptop.
 - She has never written or run Python and does not want to install anything.
 - She prefers short answers with simple analogies and no jargon.
 - She will attend an AI conference in Bengaluru next month.


**What to observe:** the LLM turns the user's eight messages into shorter, reusable facts. Mem0 stores those facts as separate memory entries. Because the write uses `infer=False`, Mem0 does not call its own large extraction prompt.


## 6.3 Search: at the start of Session 2, retrieve the user's memories

A new session, empty working memory. Before answering, we **search Mem0 with the user's question** and put the memories in the prompt.

In [17]:
found = memories(mem_search(user_question, top_k=10))

print(f"Query: {user_question}\\n")
for item in found:
    print(f"  score {item.get('score', 0):.2f} | {item['memory']}")

memory_text = "Facts about this user (tailor your answer to them):\\n" + "\\n".join(
    f"- {item['memory']}" for item in found
)

answer, tokens = ask(user_question, memory_text)
print("\\nTokens sent:", tokens)
print("Answer:", answer)

results.append({
    "experiment": "User knowledge | Mem0",
    "tokens sent": tokens,
    "check": "gave a code block: " + str("```" in answer)
})


Query: Give me code for Transformers\n
  score 0.48 | She wants a simple explanation of how ChatGPT works, especially transformers.
\nTokens sent: 114
Answer: 


**What to observe:**
- Mem0's search returned only **one** memory: the one about ChatGPT and transformers. The fact that she has never written Python was **not** retrieved.
- In this run the model's reply to that prompt came back **empty** (the check in the final table shows `False`), so this step did not show an improvement over plain RAG.
- Mem0's search is also similarity-based. Storing short, distilled facts gives a cleaner memory than raw messages, but it does not by itself guarantee that the fact that matters gets retrieved.
- What the distilled store does give us is clean, separate entries that we can update or delete (next sections).


## 6.4 Update: replace a fact when the user's situation changes

The user now says she has started learning Python. To make the update deterministic and avoid another large Mem0 extraction call, we explicitly remove the old Python-skill fact and store the new one.

This demonstrates the **Update outcome** (the old fact is replaced), but the replacement decision is made by our code rather than Mem0's automatic contradiction-resolution pipeline.


In [18]:
print("Memories BEFORE update:")
for fact in current_facts():
    print(" -", fact)

# Identify the old Python-related memory.
all_items = memories(mem_get_all())
old_python_items = [
    item for item in all_items
    if "python" in item.get("memory", "").lower()
]

# Delete old Python-related fact(s) so that they cannot contradict the new fact.
for item in old_python_items:
    print("\\nDeleting old fact:", item["memory"])
    mem.delete(item["id"])

# Store the updated fact without triggering Mem0's internal extraction prompt.
new_fact = "Riya has started learning Python, completed her first tutorial, and installed Python on her Windows laptop."
store_facts_without_inference([new_fact])

print("\\nMemories AFTER update:")
for fact in current_facts():
    print(" -", fact)


Memories BEFORE update:
 - Riya is an accountant at a mid-size firm in Pune.
 - Her team relies heavily on Excel, using pivot tables and VLOOKUP.
 - She wants a simple explanation of how ChatGPT works, especially transformers.
 - She is vegetarian.
 - She uses a Windows laptop.
 - She has never written or run Python and does not want to install anything.
 - She prefers short answers with simple analogies and no jargon.
 - She will attend an AI conference in Bengaluru next month.
\nDeleting old fact: She has never written or run Python and does not want to install anything.
\nMemories AFTER update:
 - Riya is an accountant at a mid-size firm in Pune.
 - Riya has started learning Python, completed her first tutorial, and installed Python on her Windows laptop.
 - Her team relies heavily on Excel, using pivot tables and VLOOKUP.
 - She wants a simple explanation of how ChatGPT works, especially transformers.
 - She is vegetarian.
 - She uses a Windows laptop.
 - She prefers short answers 

In [19]:
# Ask the same question again using the updated memories.
found = memories(mem_search(user_question, top_k=10))
memory_text = "Facts about this user (tailor your answer to them):\\n" + "\\n".join(
    f"- {item['memory']}" for item in found
)

answer, tokens = ask(user_question, memory_text)
print("Tokens sent:", tokens)
print("Answer:", answer)


Tokens sent: 114
Answer: Below is a **minimal, self‑contained example** that shows how a Transformer encoder can be built from scratch in PyTorch, and how you can train it on a toy dataset.  
Feel free to copy‑paste the code into a file (e.g. `simple_transformer.py`) and run it with `python simple_transformer.py`.

```python
# simple_transformer.py
# ------------------------------------------------------------
# A tiny, end‑to‑end Transformer encoder example in PyTorch.
# ------------------------------------------------------------
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# 1. Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # Create constant 'pe' matrix with values dependent on
        # position and dimension.
        pe = torch.zeros(max_len, 

**What to observe:**
- The old memory saying Riya does not know Python is gone.
- The new memory says she has started learning Python.
- The token count is unchanged (114), which suggests the search again returned only the transformers memory. This time the model did reply with code, but it is a generic PyTorch example, not one tailored to her. Retrieval, not storage, is the limit here.

This notebook performs the replacement explicitly (delete old + add new). Mem0's automatic LLM-based contradiction handling is not being tested in this low-TPM version.


## 6.5 Delete: remove a memory that is no longer wanted

First we ask a question that depends on the "vegetarian" memory. Then we delete that memory and ask again.

In [20]:
food_question = "I'm free for lunch at the conference. Suggest one dish."

def answer_with_memory(question):
    """Search Mem0, add the retrieved facts to the prompt, and answer."""
    found = memories(mem_search(question, top_k=10))
    memory_text = "Facts about this user (tailor your answer to them):\\n" + "\\n".join(
        f"- {item['memory']}" for item in found
    )
    return ask(question, memory_text)[0]

print("BEFORE delete:", answer_with_memory(food_question))

# Find and delete the vegetarian memory.
veg_results = memories(mem_search("vegetarian", top_k=10))
veg = next(
    (item for item in veg_results if "vegetarian" in item.get("memory", "").lower()),
    None
)

if veg is not None:
    print("\\nDeleting:", veg["memory"])
    mem.delete(veg["id"])
else:
    print("\\nNo vegetarian memory was found to delete.")

print("\\nMemories now:")
for fact in current_facts():
    print(" -", fact)

print("\\nAFTER delete:", answer_with_memory(food_question))


BEFORE delete: How about a **Veggie Biryani**?  
It’s like a colorful, aromatic rice‑cake that keeps you full and focused—just the right fuel for a day of AI talks.
\nDeleting: She is vegetarian.
\nMemories now:
 - Riya is an accountant at a mid-size firm in Pune.
 - Riya has started learning Python, completed her first tutorial, and installed Python on her Windows laptop.
 - Her team relies heavily on Excel, using pivot tables and VLOOKUP.
 - She wants a simple explanation of how ChatGPT works, especially transformers.
 - She uses a Windows laptop.
 - She prefers short answers with simple analogies and no jargon.
 - She will attend an AI conference in Bengaluru next month.
\nAFTER delete: Try a **Masala Dosa** – it’s like a crispy pancake filled with spicy potatoes, perfect for a quick bite.


**What to observe:** after the vegetarian fact is deleted, it should no longer appear in the user's memory list or influence answers through retrieved memory. The LLM may still independently suggest a vegetarian dish; the important check is that the deleted preference is no longer supplied as memory context.


## Troubleshooting

- **Groq 429/413 TPM error:** wait at least 60 seconds and rerun the failing cell. This notebook uses a compact extraction prompt, but previous calls in the same minute still count toward TPM.
- **Groq model unavailable:** check the currently supported model IDs in your Groq console and update the `model` value in Section 1.
- **Mem0/Qdrant file lock:** restart the Colab runtime and run cells from the beginning. Do not initialize two `Memory` objects against the same local Qdrant path simultaneously.
- **Mem0 API signature mismatch:** the helper functions in Section 6.1 try the current `filters={...}` style and fall back to the older `user_id=...` style. If your installed Mem0 version still differs, upgrade it and restart the runtime.
- **Start from a clean demo:** restart the runtime and rerun from the top. Section 6.1 removes the local `qdrant_db` directory before initializing Mem0.

**Scope note:** this notebook uses `infer=False` for Mem0 writes. Do not interpret the results as a benchmark of Mem0's built-in extraction prompt or automatic update pipeline.

# 7. Final Comparison

## 7.1 Real numbers from this notebook

The token count is the input sent for the question. Only the code-block check is automatic. Read the printed answers above for the real picture.

In [21]:
pd.DataFrame(results)


,experiment,tokens sent,check
0,World knowledge | no RAG,100,contains 'Shukla': False
1,World knowledge | RAG (top 3),382,contains 'Shukla': True
2,User knowledge | no memory,85,gave a code block: True
3,User knowledge | RAG (top 2),135,gave a code block: True
4,User knowledge | Mem0,114,gave a code block: False


## 7.2 Advantages, disadvantages and when to use each

| Approach | Advantage | Disadvantage | Use when |
|---|---|---|---|
| **No memory** | Nothing to build. | The agent knows nothing beyond its training data and nothing about the user. | One-off, generic questions. |
| **RAG for world knowledge** | Grounds answers in real, current documents. One index serves all users. | Needs documents, chunking and embeddings. Retrieval quality depends on chunking. | Shared knowledge: docs, manuals, articles. |
| **RAG for user knowledge** | Easy to reuse the same pipeline. | Retrieves by similarity to the query, so it misses facts that matter but don't look like the query. Raw messages are noisy. No update or delete. | Rarely. Not suitable for personal facts. |
| **Mem0 for user knowledge** | Stores distilled facts as separate entries. Update and Delete keep the store clean. Retrieval is still similarity-based, so in this run it did not surface the fact that mattered. | Extra LLM calls on Add. Extraction can miss or misjudge facts. Search is still similarity-based at large scale. | Personal facts that accumulate across sessions. |

## 7.3 Key takeaways

1. **World knowledge** is shared and document-like. Standard RAG works because the question looks like its answer.
2. **User knowledge** is personal and cumulative. The facts that matter often **don't look like the query**, so similarity search alone misses them.
3. **Mem0** extracts and stores distilled user facts: **Add** at session end, **Search** at session start.
4. **Update** keeps memory consistent when the user changes. **Delete** removes what should be forgotten.
5. Without memory, the agent restarts from zero in every session.